In [1]:
# !pip install -r requirements.txt
# !pip install fg-data-profiling


In [1]:

from langgraph.graph import START, END, StateGraph

import os

import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, TypedDict, List, Annotated, Any, Sequence, Literal

from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv

from data_profiling import ProfileReport


from collections import Counter as C

from langgraph.graph.message import  add_messages

from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, ChatMessage, ToolMessage, AIMessage, SystemMessage, AnyMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver


from langgraph.prebuilt import ToolNode

import json

In [3]:
load_dotenv()

AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4.1_1" #.env or this model : same :     working

# print("AZURE_OPENAI_API_VERSION : ", AZURE_OPENAI_API_VERSION)
# print("AZURE_OPENAI_DEPLOYMENT_NAME  : ", AZURE_OPENAI_DEPLOYMENT_NAME)


def setup_llm() -> AzureChatOpenAI:
    return AzureChatOpenAI(
        api_version=AZURE_OPENAI_API_VERSION,
        azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
        temperature=0.2,
    )

llm_model = setup_llm()

# single query
resp = llm_model.invoke("write 2 lines about narendra modi")
print(resp.content)


Narendra Modi is the Prime Minister of India, known for his dynamic leadership and economic reforms. He has played a significant role in shaping India's domestic and international policies.


In [3]:
df = pd.read_csv("patient_data.csv")
df.shape

(100, 18)

In [4]:

profile = ProfileReport(df, title="Patients_Data Profile Report")
profile.to_file("patients_data_profile_report.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 18/18 [00:00<00:00, 376.27it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
df.isna().sum().sum()


np.int64(0)

In [19]:
print(list(df.columns))

['patient_id', 'patient_name', 'age', 'gender', 'diagnosis', 'current_medication', 'lab_test', 'lab_value', 'lab_unit', 'last_visit_date', 'next_scheduled_visit', 'days_since_last_visit', 'missed_last_appointment', 'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2', 'notes']


In [20]:
df.head(2)

,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient_id               100 non-null    object 
 1   patient_name             100 non-null    object 
 2   age                      100 non-null    int64  
 3   gender                   100 non-null    object 
 4   diagnosis                100 non-null    object 
 5   current_medication       100 non-null    object 
 6   lab_test                 100 non-null    object 
 7   lab_value                100 non-null    float64
 8   lab_unit                 100 non-null    object 
 9   last_visit_date          100 non-null    object 
 10  next_scheduled_visit     100 non-null    object 
 11  days_since_last_visit    100 non-null    int64  
 12  missed_last_appointment  100 non-null    object 
 13  vitals_bp_systolic       100 non-null    int64  
 14  vitals_bp_diastolic      10

In [22]:
# data loading and exploratory analysis
REQUIRED = [
    "patient_id", "patient_name", "age", "gender", "diagnosis", "current_medication",
    "lab_test", "lab_value", "lab_unit", "last_visit_date", "next_scheduled_visit",
    "days_since_last_visit", "missed_last_appointment", "vitals_bp_systolic",
    "vitals_bp_diastolic", "vitals_heart_rate", "vitals_spo2", "notes"
]

assert set(REQUIRED).issubset(df.columns), "CSV does not contain all required columns"

In [23]:
assert df.patient_id.is_unique, "patient_id values must be unique"


In [24]:
display(df.isna().sum().rename("missing_values"))


patient_id                 0
patient_name               0
age                        0
gender                     0
diagnosis                  0
current_medication         0
lab_test                   0
lab_value                  0
lab_unit                   0
last_visit_date            0
next_scheduled_visit       0
days_since_last_visit      0
missed_last_appointment    0
vitals_bp_systolic         0
vitals_bp_diastolic        0
vitals_heart_rate          0
vitals_spo2                0
notes                      0
Name: missing_values, dtype: int64

In [25]:
display(df["diagnosis"].value_counts().rename("patients"))


diagnosis
Heart Failure                    12
Hypothyroidism                    9
Hypertension, Anemia              8
Type 2 Diabetes, Hypertension     7
Osteoarthritis                    7
COPD                              7
Depression                        7
Type 2 Diabetes                   6
Asthma                            6
Anemia                            6
COPD, Heart Failure               6
Hypertension, Heart Failure       5
Chronic Kidney Disease            5
Hypertension                      5
Type 2 Diabetes, CKD              4
Name: patients, dtype: int64

In [26]:
display(df["missed_last_appointment"].value_counts().rename("patients"))

missed_last_appointment
No     85
Yes    15
Name: patients, dtype: int64

In [27]:
C(df["missed_last_appointment"])

Counter({'No': 85, 'Yes': 15})

In [28]:
# Pydantic schemas: validated tool inputs and structured final response.

class PatientLookup(BaseModel):
    patient_id: str = Field(description="Patient identifier, for example P0001")

In [29]:
class PatientSearch(BaseModel):
    diagnosis : str | None = Field(default=None, description="Optional diagnosis text to match")
    note_keyword: str | None = Field(default=None, description="Optional keyword to match in notes")
    limit: int = Field(default=20, ge=1, le=50, description="Maximum records to return")

In [31]:
class PriorityAction(BaseModel):
    patient_id: str
    priority: Literal["High", "Medium", "Low"]
    evidence: list[str] = Field(description="Facts observed in the retrieved patient record")
    recommended_follow_up: list[str] = Field(description="Care-coordination and clinician-review actions only")


In [32]:
class AgentReport(BaseModel):
    summary: str
    prioritized_actions: list[PriorityAction]
    safety_note: str = Field(description="State that recommendations require clinician review and are not diagnosis/prescribing")

In [33]:
def json_records(frame: pd.DataFrame):
    return frame.to_json(orient="records", date_format="iso")


In [35]:
# deterministic data tools. The LLM decides when/how to use them and action priorities.
@tool(args_schema=PatientLookup)
def get_patient(patient_id: str) -> str:
    """Return one complete patient snapshot. Use before a patient-specific recommendation."""
    result = df[df.patient_id.str.upper() == patient_id.upper()]
    return json_records(result) if not result.empty else json.dumps({"error": "Patient not found"})


In [36]:

@tool
def get_missed_appointments() -> str:
    """Return complete records for all patients whose missed_last_appointment is Yes."""
    result = df[df.missed_last_appointment.str.strip().str.lower() == "yes"]
    return json_records(result.sort_values("days_since_last_visit", ascending=False))


In [37]:

@tool(args_schema=PatientSearch)
def search_patients(diagnosis: str | None = None, note_keyword: str | None = None, limit: int = 20) -> str:
    """Search patient records by optional diagnosis and/or clinical-note text."""
    result = df.copy()
    if diagnosis:
        result = result[result.diagnosis.str.contains(diagnosis, case=False, na=False)]
    if note_keyword:
        result = result[result.notes.str.contains(note_keyword, case=False, na=False)]
    return json_records(result.head(limit))


In [38]:

@tool
def cohort_summary() -> str:
    """Return aggregate data context. Raw lab values across different tests/units must not be compared."""
    result = {
        "patient_count": len(df),
        "missed_appointment_count": int((df.missed_last_appointment == "Yes").sum()),
        "diagnosis_counts": df.diagnosis.value_counts().to_dict(),
        "lab_test_counts": df.lab_test.value_counts().to_dict(),
        "days_since_last_visit_summary": df.days_since_last_visit.describe().round(1).to_dict(),
    }
    return json.dumps(result)

In [56]:
cohort_summary

StructuredTool(name='cohort_summary', description='Return aggregate data context. Raw lab values across different tests/units must not be compared.', args_schema=<class 'langchain_core.utils.pydantic.cohort_summary'>, func=<function cohort_summary at 0x000001FCEF4C98A0>)

In [53]:
TOOLS = [get_patient, get_missed_appointments, search_patients, cohort_summary]
TOOLS


[StructuredTool(name='get_patient', description='Return one complete patient snapshot. Use before a patient-specific recommendation.', args_schema=<class '__main__.PatientLookup'>, func=<function get_patient at 0x000001FCEF4CB4C0>),
 StructuredTool(name='get_missed_appointments', description='Return complete records for all patients whose missed_last_appointment is Yes.', args_schema=<class 'langchain_core.utils.pydantic.get_missed_appointments'>, func=<function get_missed_appointments at 0x000001FCEF4CA700>),
 StructuredTool(name='search_patients', description='Search patient records by optional diagnosis and/or clinical-note text.', args_schema=<class '__main__.PatientSearch'>, func=<function search_patients at 0x000001FCEF4CBB00>),
 StructuredTool(name='cohort_summary', description='Return aggregate data context. Raw lab values across different tests/units must not be compared.', args_schema=<class 'langchain_core.utils.pydantic.cohort_summary'>, func=<function cohort_summary at 0x0

In [52]:
agent_llm = llm_model.bind_tools(TOOLS)
agent_llm


RunnableBinding(bound=AzureChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001FCF0F7B680>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001FCF0F7BDF0>, root_client=<openai.lib.azure.AzureOpenAI object at 0x000001FCEF8B3110>, root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001FCEE49F390>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, disabled_params={'parallel_tool_calls': None}, azure_endpoint='https://rahul-manmadjt-eastus2.services.ai.azure.com/', deployment_name='gpt-4.1_1', openai_api_version='2024-12-01-preview', openai_api_type='azure'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_patient', 'description': 'Return one complete patient snapshot. Use before a patient-specific recommendation.', 'parameters': {'properties': {'patient_id': {'description': 'Patient identifier, for example P0001', 't

In [57]:
SYSTEM = """You are MediCare's care-coordination decision-support agent.
Use tools before making patient-specific conclusions. You may prioritize outreach and recommend clinician review,
but do not diagnose, prescribe, alter medication, or replace emergency services. Base evidence only on retrieved records.
Never compare raw numeric lab values between different lab tests or units. For final output, return JSON matching:
{"summary": str, "prioritized_actions": [{"patient_id": str, "priority": "High|Medium|Low",
"evidence": [str], "recommended_follow_up": [str]}], "safety_note": str}.
"""

In [65]:
# LangGraph agentic loop.
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    turns: int

In [66]:
def call_model(state: AgentState):
    response = agent_llm.invoke([SystemMessage(content=SYSTEM)] + state["messages"])
    return {"messages": [response], "turns": state["turns"] + 1}

def route(state: AgentState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None) and state["turns"] < 6:
        return "tools"
    return "finalize"

def finalize(state: AgentState):
    raw = state["messages"][-1].content
    try:
        report = AgentReport.model_validate_json(raw)
        content = report.model_dump_json(indent=2)
    except Exception:
        content = json.dumps({"summary": raw, "prioritized_actions": [], "safety_note": "Clinician review is required."})
    return {"messages": [HumanMessage(content=content)], "turns": state["turns"]}


In [68]:
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_node("finalize", finalize)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", route, {"tools": "tools", "finalize": "finalize"})
graph.add_edge("tools", "agent")
graph.add_edge("finalize", END)

In [69]:
workflow = graph.compile()


In [71]:
def run_agent(request: str) -> dict:
    result = workflow.invoke({"messages": [HumanMessage(content=request)], "turns": 0})
    return json.loads(result["messages"][-1].content)


In [73]:
# single-patient analysis
single_patient_report = run_agent(
    "Review patient P0008. Retrieve the complete record, determine follow-up priority, and return a care-coordination plan."
)
print(json.dumps(single_patient_report, indent=2))


{
  "summary": "Patient Deepa Krishnan (P0008), age 45, diagnosed with Heart Failure, is currently on Furosemide. Recent labs show elevated BNP. Vitals indicate hypertension and borderline low oxygen saturation. The patient reports dizziness and fatigue. Last visit was over 200 days ago, with the next scheduled visit in July. No missed appointments documented.",
  "prioritized_actions": [
    {
      "patient_id": "P0008",
      "priority": "High",
      "evidence": [
        "Diagnosis of Heart Failure",
        "Elevated BNP lab result",
        "Hypertension (BP 155/106)",
        "Borderline low SpO2 (92%)",
        "Symptoms: dizziness and fatigue",
        "Long interval since last visit (>200 days)"
      ],
      "recommended_follow_up": [
        "Urgent clinician review for worsening heart failure symptoms",
        "Assess medication adherence and effectiveness",
        "Evaluate for possible exacerbation or complications",
        "Consider earlier appointment or telehealt

In [77]:
#  missed-appointment follow-up
missed_appointment_report = run_agent(
    "Retrieve patients who missed their last appointment. Review the complete records and return a prioritized outreach plan for every returned patient."
)
print(json.dumps(missed_appointment_report, indent=2))

{
  "summary": "Reviewed records for patients who missed their last appointment. Several patients have poorly controlled chronic conditions, concerning symptoms, or medication adherence issues. Outreach is prioritized based on clinical urgency, symptoms, and risk factors.",
  "prioritized_actions": [
    {
      "patient_id": "P0066",
      "priority": "High",
      "evidence": [
        "Type 2 Diabetes with HbA1c 10.1%",
        "Chronic Kidney Disease",
        "Elevated BP (158/103)",
        "Missed specialist follow-up"
      ],
      "recommended_follow_up": [
        "Urgent outreach to reschedule appointment",
        "Clinician review for diabetes and CKD management",
        "Assess for symptoms and medication adherence"
      ]
    },
    {
      "patient_id": "P0043",
      "priority": "High",
      "evidence": [
        "Hypertension with BP 171/94",
        "Anemia",
        "Missed specialist follow-up"
      ],
      "recommended_follow_up": [
        "Urgent outreach 